In [0]:
countries = [
    ("USA",1000,20000),
    ("Canada",2000,30000),
    ("Mexico",3000,40000),
    ("Brazil",4000,50000),
    ("Argentina",5000,60000),
    ("India",6000,70000),
    ("China",7000,80000),
    ("Japan",8000,10000)
]

columns = ["Country","Vaccinated","Population"]

spark.createDataFrame(countries,schema=columns)\
    .write\
    .format("delta")\
    .mode("overwrite")\
    .option("enableChangeDataFeed","true")\
    .saveAsTable("practice_workspace.default.cdf_revision_silver")


In [0]:
import pyspark.sql.functions as F

spark.read.format("delta")\
    .table("practice_workspace.default.cdf_revision_silver")\
    .withColumn("Vaccinate_rate",F.col("Vaccinated")/F.col("Population"))\
    .drop("Vaccinated","Population")\
    .write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("practice_workspace.default.cdf_revision_gold")
    


In [0]:
%sql
select * from practice_workspace.default.cdf_revision_silver

- .option("enableChangeDataFeed","true") → enables CDF at read time only.
- ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed=true) → enables CDF at the table level, making change tracking persistent.


In [0]:
%sql
alter table practice_workspace.default.cdf_revision_silver set tblproperties (delta.enableChangeDataFeed = true )

In [0]:
%sql
 insert into practice_workspace.default.cdf_revision_silver values ("Australia",5000,75000)

In [0]:
%sql

update practice_workspace.default.cdf_revision_silver set Vaccinated = 6000 where Country = "Japan";

In [0]:
%sql

Delete from practice_workspace.default.cdf_revision_silver where Country = "China"

In [0]:
%sql
select * from practice_workspace.default.cdf_revision_silver

In [0]:
%sql
select * from table_changes("practice_workspace.default.cdf_revision_silver",1)

In [0]:
%sql
create or replace temporary view
Silver_new_version as 
select * from (
  select * , rank() over (partition by Country order by _commit_version desc) as rank
  from table_changes("practice_workspace.default.cdf_revision_silver",1) 
  where _change_type != "update_preimage") 
  where rank = 1

In [0]:
%sql
select * from Silver_new_version

In [0]:
%sql
select * from practice_workspace.default.cdf_revision_gold

In [0]:
%sql
Merge into practice_workspace.default.cdf_revision_gold as target
using Silver_new_version as source
on target.Country = source.Country
when matched and source._change_type = "update_postimage" then update set
target.Vaccinate_rate = source.Vaccinated/source.Population
when not matched then insert (Country,Vaccinate_rate)
values (source.Country,source.Vaccinated/source.Population)

In [0]:
%sql

select * from practice_workspace.default.cdf_revision_gold